## Neural Network

In [ ]:
import numpy as np
import polars as pl

df = pl.read_csv("dengue_features_train.csv")
labels = pl.read_csv("dengue_labels_train.csv")
df.head()

In [ ]:
labels.head()

In [ ]:
df.null_count()

In [ ]:
pixel_cols = ["ndvi_ne", "ndvi_nw", "ndvi_se", "ndvi_sw"]

df_full = df.with_columns(
    pl.col(pixel_cols)
    .fill_null(pl.col(pixel_cols).mean().over(["city", "weekofyear"]))
    .fill_null(pl.col(pixel_cols).mean().over(["city"]))
    .fill_null(pl.col(pixel_cols).mean())
)

In [ ]:
df_full.null_count()

In [ ]:
all_weather_cols = [
    "ndvi_ne", "ndvi_nw", "ndvi_se", "ndvi_sw",
    "station_max_temp_c", "station_min_temp_c", "station_avg_temp_c",
    "station_precip_mm", "station_diur_temp_rng_c", "station_avg_temp_c", 
    "reanalysis_tdtr_k", "reanalysis_specific_humidity_g_per_kg", "reanalysis_sat_precip_amt_mm", 
    "reanalysis_relative_humidity_percent", "reanalysis_precip_amt_kg_per_m2", "reanalysis_min_air_temp_k",
    "reanalysis_max_air_temp_k", "reanalysis_dew_point_temp_k", "reanalysis_avg_temp_k", "reanalysis_air_temp_k",
    "precipitation_amt_mm"
]

kelvin_cols = [
    "reanalysis_min_air_temp_k",
    "reanalysis_max_air_temp_k",
    "reanalysis_dew_point_temp_k",
    "reanalysis_avg_temp_k",
    "reanalysis_air_temp_k"
]

df_full = df_full.with_columns(
    pl.col(all_weather_cols)
    .fill_null(pl.col(all_weather_cols).mean().over(["city", "weekofyear"]))
    .fill_null(pl.col(all_weather_cols).mean().over(["city"]))
    .fill_null(pl.col(all_weather_cols).mean())
)

df_full = df_full.with_columns(
    (pl.col(kelvin_cols) - 273.15).name.suffix("_celsius")
)

df_full = df_full.drop(kelvin_cols)

In [ ]:
df_full.null_count()

In [ ]:
df_full.head()

# Cyclic encoding for weeks

In [ ]:
# https://towardsdatascience.com/cyclical-encoding-an-alternative-to-one-hot-encoding-for-time-series-features-4db46248ebba/

df_full = df_full.with_columns([
    (2 * np.pi * pl.col("weekofyear") / 52).sin().alias("week_sin"),
    (2 * np.pi * pl.col("weekofyear") / 52).cos().alias("week_cos")
])

In [ ]:
df_full.head()

In [ ]:
df_sj = df_full.filter(pl.col("city") == "sj")
df_iq = df_full.filter(pl.col("city") == "iq")

print(f"San Juan: {len(df_sj)} rows")
print(f"Iquitos: {len(df_iq)} rows")

In [ ]:
labels.head()

In [ ]:
exclude_cols = ["city", "year", "weekofyear", "week_start_date", "total_cases"]
feature_cols = [c for c in df_full.columns if c not in exclude_cols]

labels_sj = labels.filter(pl.col("city") == "sj")
labels_iq = labels.filter(pl.col("city") == "iq")

In [ ]:

X_sj = df_sj.select(feature_cols).to_numpy().astype(np.float32)
X_iq = df_iq.select(feature_cols).to_numpy().astype(np.float32)

y_sj = labels_sj["total_cases"].to_numpy().astype(np.float32)
y_iq = labels_iq["total_cases"].to_numpy().astype(np.float32)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_sj, X_val_sj, y_train_sj, y_val_sj = train_test_split(X_sj, y_sj, test_size=0.2, random_state=42)
scaler_sj = StandardScaler()
X_train_sj = scaler_sj.fit_transform(X_train_sj)
X_val_sj = scaler_sj.transform(X_val_sj)

X_train_iq, X_val_iq, y_train_iq, y_val_iq = train_test_split(X_iq, y_iq, test_size=0.2, random_state=42)
scaler_iq = StandardScaler()
X_train_iq = scaler_iq.fit_transform(X_train_iq)
X_val_iq = scaler_iq.transform(X_val_iq)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# San Juan
train_ds_sj = TensorDataset(torch.tensor(X_train_sj), torch.tensor(y_train_sj).unsqueeze(1))
train_loader_sj = DataLoader(train_ds_sj, batch_size=32, shuffle=True)

# Iquitos
train_ds_iq = TensorDataset(torch.tensor(X_train_iq), torch.tensor(y_train_iq).unsqueeze(1))
train_loader_iq = DataLoader(train_ds_iq, batch_size=32, shuffle=True)

In [ ]:
class DengueNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.net(x)

model = DengueNN(X_train_scaled.shape[1])

In [ ]:
# San Juan model
model_sj = DengueNN(X_train_sj.shape[1])

# Iquitos model
model_iq = DengueNN(X_train_iq.shape[1])

In [ ]:
optimizer_sj = torch.optim.Adam(model_sj.parameters(), lr=0.001)
loss_func = nn.L1Loss()

print("Training San Juan model")
for epoch in range(80):
    model_sj.train()
    for X_batch, y_batch in train_loader_sj:
        optimizer_sj.zero_grad()
        pred = model_sj(X_batch)
        loss = loss_func(pred, y_batch)
        loss.backward()
        optimizer_sj.step()
    
    if (epoch + 1) % 10 == 0:
        model_sj.eval()
        with torch.no_grad():
            val_preds = model_sj(torch.tensor(X_val_sj))
            val_loss = loss_func(val_preds, torch.tensor(y_val_sj).unsqueeze(1))
        print(f"Epoch {epoch+1}, Val MSE: {val_loss.item():.2f}")

optimizer_iq = torch.optim.Adam(model_iq.parameters(), lr=0.001)

print("\nTraining Iquitos model")
for epoch in range(100):
    model_iq.train()
    for X_batch, y_batch in train_loader_iq:
        optimizer_iq.zero_grad()
        pred = model_iq(X_batch)
        loss = loss_func(pred, y_batch)
        loss.backward()
        optimizer_iq.step()
    
    if (epoch + 1) % 10 == 0:
        model_iq.eval()
        with torch.no_grad():
            val_preds = model_iq(torch.tensor(X_val_iq))
            val_loss = loss_func(val_preds, torch.tensor(y_val_iq).unsqueeze(1))
        print(f"Epoch {epoch+1}, Val MAE: {val_loss.item():.2f}")

# Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Train San Juan model
lr_sj = LinearRegression()
lr_sj.fit(X_train_sj, y_train_sj)

# Train Iquitos model
lr_iq = LinearRegression()
lr_iq.fit(X_train_iq, y_train_iq)

# Evaluate
preds_sj = lr_sj.predict(X_val_sj)
preds_iq = lr_iq.predict(X_val_iq)

print(f"San Juan MAE: {mean_absolute_error(y_val_sj, preds_sj):.2f}")
print(f"Iquitos MAE: {mean_absolute_error(y_val_iq, preds_iq):.2f}")

In [ ]:
from sklearn.metrics import mean_absolute_error

# San Juan
model_sj.eval()
with torch.no_grad():
    preds_sj = model_sj(torch.tensor(X_val_sj)).numpy().flatten()
print(f"San Juan MAE: {mean_absolute_error(y_val_sj, preds_sj):.2f}")

# Iquitos
model_iq.eval()
with torch.no_grad():
    preds_iq = model_iq(torch.tensor(X_val_iq)).numpy().flatten()
print(f"Iquitos MAE: {mean_absolute_error(y_val_iq, preds_iq):.2f}")

## Test

In [ ]:
test_df = pl.read_csv('dengue_features_test.csv')

pixel_cols = ["ndvi_ne", "ndvi_nw", "ndvi_se", "ndvi_sw"]

test_df = test_df.with_columns(
    pl.col(pixel_cols)
    .fill_null(pl.col(pixel_cols).mean().over(["city", "weekofyear"]))
    .fill_null(pl.col(pixel_cols).mean().over(["city"]))
    .fill_null(pl.col(pixel_cols).mean())
)

all_weather_cols = [
    "ndvi_ne", "ndvi_nw", "ndvi_se", "ndvi_sw",
    "station_max_temp_c", "station_min_temp_c", "station_avg_temp_c",
    "station_precip_mm", "station_diur_temp_rng_c", "station_avg_temp_c",
    "reanalysis_tdtr_k", "reanalysis_specific_humidity_g_per_kg", "reanalysis_sat_precip_amt_mm",
    "reanalysis_relative_humidity_percent", "reanalysis_precip_amt_kg_per_m2", "reanalysis_min_air_temp_k",
    "reanalysis_max_air_temp_k", "reanalysis_dew_point_temp_k", "reanalysis_avg_temp_k", "reanalysis_air_temp_k",
    "precipitation_amt_mm"
]

kelvin_cols = [
    "reanalysis_min_air_temp_k",
    "reanalysis_max_air_temp_k",
    "reanalysis_dew_point_temp_k",
    "reanalysis_avg_temp_k",
    "reanalysis_air_temp_k"
]

test_df = test_df.with_columns(
    pl.col(all_weather_cols)
    .fill_null(pl.col(all_weather_cols).mean().over(["city", "weekofyear"]))
    .fill_null(pl.col(all_weather_cols).mean().over(["city"]))
    .fill_null(pl.col(all_weather_cols).mean())
)

test_df = test_df.with_columns(
    (pl.col(kelvin_cols) - 273.15).name.suffix("_celsius")
)

test_df = test_df.drop(kelvin_cols)

test_df = test_df.with_columns([
    (2 * np.pi * pl.col("weekofyear") / 52).sin().alias("week_sin"),
    (2 * np.pi * pl.col("weekofyear") / 52).cos().alias("week_cos")
])

test_sj = test_df.filter(pl.col("city") == "sj")
test_iq = test_df.filter(pl.col("city") == "iq")

X_test_sj = test_sj.select(feature_cols).to_numpy().astype(np.float32)
X_test_iq = test_iq.select(feature_cols).to_numpy().astype(np.float32)

X_test_sj_scaled = scaler_sj.transform(X_test_sj)
X_test_iq_scaled = scaler_iq.transform(X_test_iq)

ids_sj = test_sj.select(["city", "year", "weekofyear"])
ids_iq = test_iq.select(["city", "year", "weekofyear"])

In [ ]:
model_sj.eval()
model_iq.eval()

with torch.no_grad():
    preds_sj = model_sj(torch.tensor(X_test_sj_scaled)).numpy().flatten()
    preds_iq = model_iq(torch.tensor(X_test_iq_scaled)).numpy().flatten()

preds_sj = np.clip(np.round(preds_sj), 0, None).astype(int)
preds_iq = np.clip(np.round(preds_iq), 0, None).astype(int)

results_sj = ids_sj.to_pandas()
results_sj['total_cases'] = preds_sj

results_iq = ids_iq.to_pandas()
results_iq['total_cases'] = preds_iq

results = pd.concat([results_sj, results_iq])
results.to_csv('predictions.csv', index=False)

print(f"Saved {len(results)} predictions")
print("Sample predictions:")
print(results.head())

In [ ]:
print(f"Scaler expects: {scaler_sj.n_features_in_} features")
print(f"Test has: {X_test_sj.shape[1]} features")

# LinearRegression predictions

In [ ]:
test = pd.read_csv('dengue_features_test.csv')

test_sj = test[test['city'] == 'sj']
test_iq = test[test['city'] == 'iq']

ids_sj = test_sj[['city', 'year', 'weekofyear']]
ids_iq = test_iq[['city', 'year', 'weekofyear']]

drop_cols = ['city', 'year', 'weekofyear', 'week_start_date']
X_test_sj = test_sj.drop(columns=drop_cols).fillna(0).to_numpy()
X_test_iq = test_iq.drop(columns=drop_cols).fillna(0).to_numpy()


preds_sj = lr_sj.predict(X_test_sj_scaled)
preds_iq = lr_iq.predict(X_test_iq_scaled)


preds_sj = np.clip(np.round(preds_sj), 0, None).astype(int)
preds_iq = np.clip(np.round(preds_iq), 0, None).astype(int)

results_sj = ids_sj.copy()
results_sj['total_cases'] = preds_sj

results_iq = ids_iq.copy()
results_iq['total_cases'] = preds_iq

results = pd.concat([results_sj, results_iq])
results.to_csv('predictions_lr.csv', index=False)

print(f"Saved {len(results)} predictions")